# 01. 画像劣化のメカニズムと古典的復元（Wiener Filter・補間拡大）

**セミナー対応章**：第1章（1）〜（3）　｜　**推奨ランタイム**：CPUでも可（Colab無料枠でOK）　｜　**所要時間**：約2分

このノートブックでは、
1. 劣化モデル $y = (x * k)\downarrow_s + n$（ボケ・縮小・ノイズ）で劣化画像を作る
2. 古典的な復元：補間拡大（Nearest / Bicubic / Lanczos）と Wiener Filter
3. 「失われた情報は戻らない」ことを周波数スペクトルで確認する

を体験します。生成AIを使う前に「何が難しいのか」を理解するための出発点です。

In [ ]:
# ===== 共通セットアップ（このセルを最初に1回だけ実行してください） =====
import os, sys, subprocess, time, urllib.request

IN_COLAB = "google.colab" in sys.modules

def pip_install(*pkgs):
    """足りないライブラリを静かにインストールする（Colab用）"""
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

if IN_COLAB:
    # Colabには torch / transformers / diffusers などが最初から入っています。
    # 不足分だけ追加します（1〜2分）。
    pip_install("spandrel", "lpips", "diffusers>=0.30", "accelerate", "peft")

import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
print(f"実行環境: {'Google Colab' if IN_COLAB else 'ローカルPC'} / torch {torch.__version__}")
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}  VRAM: {p.total_memory / 1024**3:.1f} GB")
else:
    print("GPUが見つかりません。Colabでは［ランタイム］→［ランタイムのタイプを変更］→ T4 GPU を選んでください。")

WEIGHTS_DIR = os.path.abspath("weights")
os.makedirs(WEIGHTS_DIR, exist_ok=True)

def download(url, name=None):
    """学習済み重みをダウンロード（2回目以降はキャッシュを使う）"""
    path = os.path.join(WEIGHTS_DIR, name or url.split("/")[-1])
    if not os.path.exists(path):
        print("downloading", url)
        urllib.request.urlretrieve(url, path + ".part")      # 途中で止めても壊れたファイルが残らないように
        os.replace(path + ".part", path)
    return path

def load_sample(name="astronaut", size=None):
    """scikit-image同梱のサンプル画像（ネット不要）。name: astronaut / coffee / chelsea / rocket"""
    from skimage import data
    img = Image.fromarray(getattr(data, name)()).convert("RGB")
    if size:
        img = img.resize((size, size) if isinstance(size, int) else size, Image.BICUBIC)
    return img

def load_portrait():
    """顔が大きく写った 512x512 の高画質画像（Kodak 標準テスト画像 kodim04 の上部を切り出し）。
    Kodak 画像は画像処理の評価用に広く使われているフリーのテスト画像（r0k.us/graphics/kodak/）。
    ダウンロードできない場合は astronaut で代用。"""
    try:
        img = Image.open(download("https://r0k.us/graphics/kodak/kodak/kodim04.png")).convert("RGB")
        return img.crop((0, 160, 512, 672))
    except Exception as e:
        print("kodim04 を取得できなかったため astronaut を使います:", e)
        return load_sample("astronaut")

def to_np(img):
    """PIL → float32 [0,1] の (H,W,3) 配列"""
    return np.asarray(img.convert("RGB"), dtype=np.float32) / 255.0

def to_pil(arr):
    return Image.fromarray((np.clip(arr, 0, 1) * 255 + 0.5).astype(np.uint8))

def to_tensor(img):
    """PIL / ndarray → (1,3,H,W) テンソル"""
    a = to_np(img) if isinstance(img, Image.Image) else img
    return torch.from_numpy(np.ascontiguousarray(a, dtype=np.float32)).permute(2, 0, 1)[None].contiguous()

def tensor_to_pil(t):
    return to_pil(t[0].detach().float().clamp(0, 1).permute(1, 2, 0).cpu().numpy())

def psnr(a, b):
    a, b = to_np(a) if isinstance(a, Image.Image) else a, to_np(b) if isinstance(b, Image.Image) else b
    return 10 * np.log10(1.0 / np.mean((a - b) ** 2))

def ssim(a, b):
    from skimage.metrics import structural_similarity
    a, b = to_np(a) if isinstance(a, Image.Image) else a, to_np(b) if isinstance(b, Image.Image) else b
    return structural_similarity(a, b, channel_axis=2, data_range=1.0)

def show(images, titles=None, size=4, rows=1):
    """画像を横に並べて表示（タイトルは文字化け防止のため英数字推奨）"""
    n = len(images)
    cols = (n + rows - 1) // rows
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows), squeeze=False)
    for i, ax in enumerate(axes.flat):
        ax.axis("off")
        if i < n:
            im = images[i]
            if isinstance(im, np.ndarray) and im.ndim == 2:
                ax.imshow(im, cmap="gray")                      # 2次元配列（マップ・スペクトル）は自動スケール
            else:
                if isinstance(im, np.ndarray) and im.dtype != np.uint8:
                    im = np.clip(im, 0, 1)                      # float 画像は 0〜1 に収める
                ax.imshow(im)
            if titles:
                ax.set_title(titles[i], fontsize=11)
    plt.tight_layout()
    plt.show()

def gpu_mem():
    if DEVICE == "cuda":
        print(f"GPUメモリ使用量(ピーク): {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

## 1. 劣化モデルで「劣化画像」を作る
実世界の劣化は、**ボケ（PSF：点拡がり関数 k との畳み込み）→ 縮小（↓s）→ ノイズ（n）→ JPEG圧縮** の組み合わせとして近似できます。

In [ ]:
import cv2

def gaussian_psf(size=15, sigma=2.0):
    ax = np.arange(size) - size // 2
    g = np.exp(-ax**2 / (2 * sigma**2))
    k = np.outer(g, g)
    return k / k.sum()

def motion_psf(length=15, angle=30):
    k = np.zeros((length, length), np.float32)
    k[length // 2, :] = 1.0
    M = cv2.getRotationMatrix2D((length / 2 - 0.5, length / 2 - 0.5), angle, 1.0)
    k = cv2.warpAffine(k, M, (length, length))
    return k / k.sum()

def blur(x, k):
    return cv2.filter2D(x, -1, k, borderType=cv2.BORDER_REFLECT)

def jpeg(x, q=30):
    ok, buf = cv2.imencode(".jpg", (np.clip(x, 0, 1) * 255).astype(np.uint8)[..., ::-1],
                           [cv2.IMWRITE_JPEG_QUALITY, q])
    return cv2.imdecode(buf, 1)[..., ::-1].astype(np.float32) / 255

rng = np.random.default_rng(0)
x = to_np(load_sample("astronaut"))           # 正解画像（512x512）
k = gaussian_psf(15, 2.0)

y_blur = blur(x, k)
y_noise = np.clip(x + rng.normal(0, 0.05, x.shape), 0, 1)
y_down = cv2.resize(x, (128, 128), interpolation=cv2.INTER_AREA)
y_jpeg = jpeg(x, 10)
y_all = jpeg(np.clip(cv2.resize(blur(x, k), (128, 128), interpolation=cv2.INTER_AREA)
                     + rng.normal(0, 0.03, (128, 128, 3)), 0, 1), 40)

show([x, y_blur, y_noise, y_down, y_jpeg, y_all],
     ["Ground truth", "Blur (Gaussian PSF)", "Noise (sigma=0.05)",
      "Downscale x1/4", "JPEG q=10", "Blur+Down+Noise+JPEG"], size=3.2, rows=2)

## 2. 補間による拡大 vs 正解
補間（Nearest / Bilinear / Bicubic / Lanczos）は **周辺画素の重み付き平均** なので、新しい高周波（細かいテクスチャ）は作れません。

In [ ]:
small = cv2.resize(x, (128, 128), interpolation=cv2.INTER_AREA)
ups = {
    "Nearest": cv2.resize(small, (512, 512), interpolation=cv2.INTER_NEAREST),
    "Bilinear": cv2.resize(small, (512, 512), interpolation=cv2.INTER_LINEAR),
    "Bicubic": cv2.resize(small, (512, 512), interpolation=cv2.INTER_CUBIC),
    "Lanczos": cv2.resize(small, (512, 512), interpolation=cv2.INTER_LANCZOS4),
}
crop = (slice(80, 240), slice(160, 320))   # 顔付近を拡大表示
show([x[crop]] + [np.clip(v[crop], 0, 1) for v in ups.values()],
     ["Ground truth"] + [f"{k}  PSNR={psnr(np.clip(v,0,1), x):.2f}dB" for k, v in ups.items()], size=3.2)

## 3. Wiener Filter によるボケ除去
周波数領域で
$$\hat X(u,v) = \frac{K^*(u,v)}{|K(u,v)|^2 + \lambda}\,Y(u,v)$$
とする古典的な手法です。$\lambda$（ノイズ対信号比）が小さいとノイズが爆発し、大きいとボケが残ります。

In [ ]:
def psf2otf(k, shape):
    pad = np.zeros(shape, np.float32)
    kh, kw = k.shape
    pad[:kh, :kw] = k
    pad = np.roll(pad, (-(kh // 2), -(kw // 2)), axis=(0, 1))
    return np.fft.fft2(pad)

def wiener(y, k, lam):
    K = psf2otf(k, y.shape[:2])
    H = np.conj(K) / (np.abs(K) ** 2 + lam)
    return np.stack([np.real(np.fft.ifft2(np.fft.fft2(y[..., c]) * H)) for c in range(3)], -1)

y = np.clip(blur(x, k) + rng.normal(0, 0.01, x.shape), 0, 1)   # ボケ + 弱いノイズ
results = [wiener(y, k, lam) for lam in (1e-4, 1e-2, 1e-1)]
show([x, y] + results,
     ["Ground truth", f"Blurred+noise {psnr(y,x):.2f}dB"]
     + [f"Wiener lam={l}  {psnr(np.clip(r,0,1),x):.2f}dB" for l, r in zip((1e-4, 1e-2, 1e-1), results)],
     size=3.2)

## 4. 「失われた情報」を周波数で見る
PSF の周波数応答 |K| がほぼ 0 になる帯域では、どんなフィルタでも情報を取り戻せません。
縮小（ダウンサンプリング）でも、ナイキスト周波数より上の成分は消えてしまいます。
→ ここを **学習データから「予測」して埋める** のが生成AIのアプローチです。

In [ ]:
def log_spectrum(img):
    g = img.mean(-1)
    return np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(g))))

K = np.abs(np.fft.fftshift(psf2otf(k, x.shape[:2])))
show([log_spectrum(x), log_spectrum(y), K, log_spectrum(np.clip(ups["Bicubic"], 0, 1))],
     ["Spectrum: GT", "Spectrum: blurred", "|K| of PSF (0=lost)", "Spectrum: bicubic x4"], size=3.5)

## 5. Richardson–Lucy 法（反復型デコンボリューション）
天文学や顕微鏡で使われてきた反復法。反復回数を増やすとシャープになりますが、ノイズやリンギングも増えます。

In [ ]:
from skimage.restoration import richardson_lucy

rl = {n: np.stack([richardson_lucy(y[..., c], k, num_iter=n) for c in range(3)], -1) for n in (5, 30, 100)}
show([y] + list(rl.values()), ["Input"] + [f"RL iter={n}  {psnr(np.clip(v,0,1),x):.2f}dB" for n, v in rl.items()], size=3.5)

## まとめ
- 劣化は「ボケ・縮小・ノイズ・圧縮」の合成としてモデル化できる
- 補間や Wiener Filter は **観測された情報の範囲内** でしか復元できない
- 消えた高周波を取り戻すには「自然画像らしさ」の知識（事前分布）が必要 → 次のノートブック以降で生成AIを使います

**自分の画像で試す**：`x = to_np(Image.open("my_image.jpg").convert("RGB"))` に置き換えて再実行してください（Colabでは左のフォルダアイコンからアップロード）。